# Vélib' processing: bronze → silver → gold

Run by ADF after each ingestion, or by hand (*Run all*).

- **silver**: one row per station and snapshot, appended incrementally (watermark on `ingested_at`) with a Delta `MERGE` on (`station_id`, `ingested_at`), so replaying bronze never creates duplicates.
- **gold**: KPI tables rebuilt from silver on each run (small, idempotent).

The transformations live in `src/data_platform_project/transform.py` (unit-tested locally) and are loaded from the lake with `sc.addPyFile`.

In [ ]:
# Parameters (overridable from the ADF Synapse notebook activity)
storage_account = "stvelibmwzyy4"
transform_lib = f"abfss://synapse@{storage_account}.dfs.core.windows.net/libs/transform.py"

In [ ]:
import time
from datetime import datetime, timezone

from delta.tables import DeltaTable

start = time.time()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", "8")

sc.addPyFile(transform_lib)
import transform as tr

BRONZE = f"abfss://bronze@{storage_account}.dfs.core.windows.net"
SILVER = f"abfss://silver@{storage_account}.dfs.core.windows.net/velib/station_status"
GOLD = f"abfss://gold@{storage_account}.dfs.core.windows.net/velib"

## Bronze → silver (incremental)

In [ ]:
watermark = None
if DeltaTable.isDeltaTable(spark, SILVER):
    watermark = tr.max_ingested_at(spark.read.format("delta").load(SILVER))
print(f"Watermark: {watermark}")

status_raw = tr.read_bronze(spark, BRONZE, "station_status")
status_new = tr.new_since(
    status_raw.withColumn("ingested_at", tr.ingested_at_from_path(status_raw.source_file)),
    watermark,
)
information_raw = tr.read_bronze(spark, BRONZE, "station_information")

silver_new = tr.to_silver(status_new, information_raw).cache()
new_rows = silver_new.count()
print(f"New silver rows: {new_rows}")

In [ ]:
if new_rows:
    if DeltaTable.isDeltaTable(spark, SILVER):
        (
            DeltaTable.forPath(spark, SILVER).alias("t")
            .merge(
                silver_new.alias("s"),
                "t.station_id = s.station_id AND t.ingested_at = s.ingested_at",
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        silver_new.write.format("delta").partitionBy("snapshot_date").save(SILVER)
silver_new.unpersist()

## Silver → gold

In [ ]:
silver = spark.read.format("delta").load(SILVER).cache()
series = tr.station_timeseries(silver).cache()
latest_snapshot = series.agg({"ingested_at": "max"}).first()[0]

gold_tables = {
    "network_snapshot": tr.network_snapshot(silver),
    "zone_snapshot": tr.zone_snapshot(silver),
    "station_timeseries": series,
    "station_latest": series.filter(series.ingested_at == latest_snapshot),
}
for name, df in gold_tables.items():
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(f"{GOLD}/{name}")
    )
    print(f"gold/{name}: {spark.read.format('delta').load(f'{GOLD}/{name}').count()} rows")

## Run log

In [ ]:
now = datetime.now(timezone.utc).replace(tzinfo=None)
print(f"Silver rows total : {silver.count()}")
print(f"Snapshots         : {silver.select('ingested_at').distinct().count()}")
print(f"Latest snapshot   : {latest_snapshot} UTC")
print(f"Latency           : {(now - latest_snapshot).total_seconds() / 60:.1f} min (snapshot -> gold)")
print(f"Duration          : {time.time() - start:.0f} s")
display(gold_tables["station_latest"].groupBy("anomaly").count())